# Negotiate, Then Solve: training the 4B local model on two T4s

LoRA fine-tuning of **Qwen3.5-4B** with plain Hugging Face (transformers, PEFT,
TRL), no Unsloth. The data, the prompt rendering and the quick check are the same as
`train_compiler_kaggle.ipynb`, so the model drops into `--local` the same way.

**Why not Unsloth.** A T4 has no bf16 and Qwen3.5 does not train in fp16, so the
weights stay in float32: about 17 GB for the 4B model, more than one T4 (15 GB).
Unsloth trains on one GPU only, and ordinary multi-GPU training (DDP) copies the whole
model to each GPU, so neither fits. Here `device_map="auto"` splits the layers across
both T4s (about 8 GB each). Only one GPU computes at a time, so the second one adds
memory, not speed: expect roughly 2-3x the time of the 2B run.

**Before running**

1. Upload `data/multitask/` (or `data/multitask.zip`) as a Kaggle dataset with
   *multitask* in its name and add it (*Add Input*). Remove older versions of it.
2. *Settings*: **Accelerator: GPU T4 x2**, **Internet: on**.
3. Run with `TRIAL = True` (20 steps). The training cell prints the seconds per step
   and the projected length of the full run, against Kaggle's 12-hour session limit.
4. Full run: set `TRIAL = False`, then *Save Version* > *Save & Run All (Commit)*.
   A commit runs in the background for up to 12 hours and keeps `/kaggle/working`.
   Training saves a checkpoint every 50 steps and stops itself (with a checkpoint)
   after `TIME_BUDGET_H` hours. If it stopped early, add that version's output as an
   input and commit again: it resumes from the latest checkpoint (`RESUME = True`).

**fp16.** `PRECISION = "fp16"` keeps the frozen weights in fp16 and the adapters in
float32 (mixed precision): about half the memory and several times faster on a T4.
Qwen3.5's linear-attention layers may overflow in fp16; the training cell stops if the
loss is not finite. Try it in a TRIAL before committing hours to it.

In [ ]:
%%capture
FAST_KERNELS = False  # True: also install flash-linear-attention (Triton kernels for the linear-attention layers; may not support the T4)
# Keep Kaggle's CUDA build of torch: a dependency must not swap it for another build.
!pip freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/torch-pin.txt
!pip install --upgrade -c /tmp/torch-pin.txt "transformers>=5" peft trl accelerate datasets
# Kaggle ships torchao 0.10; PEFT refuses torchao < 0.16 when it adds LoRA layers. Nothing here uses it.
!pip uninstall -y -q torchao
if FAST_KERNELS:
    !pip install flash-linear-attention

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, transformers, peft, trl

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, (
    "Qwen3.5 needs transformers v5. Restart the kernel (Run > Restart) and run from this cell.")
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, (
    f"no GPU (torch {torch.__version__}, CUDA {torch.version.cuda}): this session runs on Kaggle's CPU image. "
    "Set Settings > Accelerator > GPU T4 x2 for this notebook (and check it again in Save Version). "
    "If it is set, another GPU session (e.g. the 2B run) may be using your GPU: wait for it to finish.")
for i in range(N_GPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")
if N_GPU < 2:
    print("WARNING: one GPU. Set Accelerator: GPU T4 x2; the 4B model in float32 does not fit on one T4.")

## Settings

In [ ]:
import glob, math, re
from pathlib import Path

MODEL = "Qwen/Qwen3.5-4B"
TRIAL = True                # True: 20 steps, time projection, no export
PRECISION = "fp32"          # "fp32" (safe on a T4) or "fp16" (faster; test in a TRIAL first)
EPOCHS = 3
LR = 2e-4
MAX_LEN = 2048              # policy prompts reach ~1600 tokens
BATCH, ACCUM = 1, 16        # effective batch 16, as in the 2B run
SEED = 3407
# Weights per GPU (the model is ~17 GB in float32). The output layer is tied to the
# embeddings, so it sits on GPU 0; TRL's default loss ("chunked_nll") projects only the
# answer tokens, in chunks, so the 248k-token logits stay small. If the load cell reports
# CPU offload, raise these; if GPU 0 runs out of memory in training, move 1-2 GB to GPU 1.
MAX_MEMORY = {0: "8GiB", 1: "12GiB"}
TIME_BUDGET_H = 11.0        # stop with a checkpoint before Kaggle's 12-hour limit
RESUME = True               # continue from the latest checkpoint in an attached earlier output

found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
assert found, "no train.jsonl under /kaggle/input: add the multitask dataset (Add Input)"
multitask = [f for f in found if "multitask" in f.lower()]
if not multitask and len(found) > 1:
    print("WARNING: no dataset path contains 'multitask'; using", found[0], "of", found)
DATA = Path((multitask or found)[0]).parent
NAME = MODEL.split("/")[-1].lower() + "-nts"
OUT = Path("/kaggle/working") / NAME
print("data:", DATA, "| output:", OUT)

## Model and LoRA adapters

Qwen3.5 is a vision-language model (`Qwen3_5ForConditionalGeneration`). The whole
model is loaded, so the merged checkpoint keeps its architecture for the GGUF
conversion, but LoRA goes on the language layers only (the same modules as the 2B
run) and the vision tower stays frozen and unused.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import LoraConfig, get_peft_model

DTYPE = torch.float32 if PRECISION == "fp32" else torch.float16
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.chat_template is None:  # some releases keep the template with the processor
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODEL).chat_template

model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype=DTYPE, device_map="auto", max_memory=MAX_MEMORY, attn_implementation="sdpa")
placed = {str(d) for d in model.hf_device_map.values()}
print("devices:", sorted(placed))
assert not placed & {"cpu", "disk"}, (
    "part of the model was offloaded to the CPU (very slow): raise MAX_MEMORY or use PRECISION = 'fp16'")
model.config.use_cache = False
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GB of weights")

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"))
model.print_trainable_parameters()
print("eos:", repr(tok.eos_token))

## Data

The prompt is rendered with the chat template and **thinking disabled**, exactly as
the local server renders it, and loss is computed on the JSON answer only.

In [ ]:
import json
from collections import Counter
from datasets import load_dataset

def render_prompt(messages):
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)

def to_pc(row):
    return {"prompt": render_prompt(row["messages"]),
            "completion": row["messages"][2]["content"] + tok.eos_token}

raw = load_dataset("json", data_files={"train": str(DATA / "train.jsonl"), "val": str(DATA / "val.jsonl")})
if "task" in raw["train"].column_names:
    print("train tasks:", Counter(raw["train"]["task"]), "| val tasks:", Counter(raw["val"]["task"]))
ds = raw.map(to_pc, remove_columns=raw["train"].column_names)
print(ds)
print(ds["train"][0]["prompt"][-400:])
print(ds["train"][0]["completion"])
n_tokens = [len(tok(p + c)["input_ids"]) for p, c in zip(ds["train"]["prompt"], ds["train"]["completion"])]
print("tokens per example: max", max(n_tokens), "mean", sum(n_tokens) // len(n_tokens))
assert max(n_tokens) <= MAX_LEN, "raise MAX_LEN"
STEPS_PER_EPOCH = math.ceil(len(ds["train"]) / (BATCH * ACCUM))
print("optimizer steps:", STEPS_PER_EPOCH, "per epoch,", STEPS_PER_EPOCH * EPOCHS, "in all")

## Train

In [ ]:
import time
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

class Clock(TrainerCallback):
    """Seconds per optimizer step, a stop with a checkpoint before the session
    limit, and a stop if the loss is not finite (fp16 overflow)."""
    def __init__(self):
        self.start, self.marks = time.time(), []
    def on_step_end(self, args, state, control, **kwargs):
        self.marks.append(time.time())
        if not TRIAL and (time.time() - self.start) / 3600 > TIME_BUDGET_H:
            print(f"time budget reached at step {state.global_step}: saving a checkpoint and stopping")
            control.should_save = control.should_training_stop = True
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and not math.isfinite(logs["loss"]):
            print("loss is not finite (fp16 overflows here): use PRECISION = 'fp32'")
            control.should_training_stop = True
    def per_step(self):
        m = self.marks[5:] if len(self.marks) > 8 else self.marks  # skip warm-up steps
        return (m[-1] - m[0]) / max(len(m) - 1, 1)

resume = None
if RESUME and not TRIAL:
    states = glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
    if states:
        resume = str(Path(max(states, key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))).parent)
print("resume from:", resume)

clock = Clock()
trainer = SFTTrainer(
    model = model,
    processing_class = tok,
    train_dataset = ds["train"],
    eval_dataset = ds["val"],
    callbacks = [clock],
    args = SFTConfig(
        output_dir = str(OUT / "checkpoints"),
        max_length = MAX_LEN,
        completion_only_loss = True,          # learn the JSON, not the prompt
        per_device_train_batch_size = BATCH,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = ACCUM,
        num_train_epochs = EPOCHS,
        max_steps = 20 if TRIAL else -1,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_steps = 5,
        weight_decay = 0.01,
        optim = "adamw_torch",                # only the adapters train: the optimizer state is small
        fp16 = PRECISION == "fp16",
        bf16 = False,                         # T4 has no bf16
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        logging_steps = 5,
        eval_strategy = "no" if TRIAL else "epoch",
        save_strategy = "no" if TRIAL else "steps",
        save_steps = 50,
        save_total_limit = 2,
        report_to = "none",
        seed = SEED,
    ),
)
print("model parallel across GPUs:", trainer.is_model_parallel)
stats = trainer.train(resume_from_checkpoint=resume)
print(f"{stats.metrics['train_runtime'] / 60:.1f} min this session")
for i in range(N_GPU):
    print(f"GPU {i}: peak {torch.cuda.max_memory_allocated(i) / 2**30:.1f} GB")

per_step = clock.per_step()
total = STEPS_PER_EPOCH * EPOCHS
hours = per_step * total / 3600
print(f"{per_step:.0f} s per optimizer step -> the full run ({total} steps, {EPOCHS} epochs) takes about {hours:.1f} h"
      " plus evaluation")
if hours > TIME_BUDGET_H:
    fit = int(TIME_BUDGET_H * 3600 / per_step / STEPS_PER_EPOCH)
    print(f"more than one {TIME_BUDGET_H:.0f}-hour session: commit twice (it resumes), "
          f"or set EPOCHS = {max(fit, 1)} to fit in one" + ("" if fit else " (one epoch already needs two sessions)"))

## Quick check on the validation split

Greedy decoding on validation rows: is the output valid JSON, and is it identical to
the gold answer (per task)? Explanations are free text, so for them only the cited
fact IDs are compared. A sanity check only; the real scores come from the project's
evaluations with `--local`.

In [ ]:
from collections import defaultdict

model.eval()
PER_TASK = 2 if TRIAL else 15  # greedy decoding in float32 across two GPUs is slow
val_rows, taken = [], defaultdict(int)
for line in open(DATA / "val.jsonl", encoding="utf-8"):
    row = json.loads(line)
    task = row.get("task", "compile")
    if taken[task] < PER_TASK:
        taken[task] += 1
        val_rows.append(row)

def norm(o):
    """Drop empty/default fields: [] vs missing, "none" vs missing mean the same constraint."""
    if isinstance(o, dict):
        return {k: norm(v) for k, v in o.items() if v not in (None, [], "none")}
    if isinstance(o, list):
        return [norm(x) for x in o]
    return o

def key(task, o):
    if task == "explain":  # compare what is cited, not the wording
        return sorted({i for c in o.get("claims", []) for i in c.get("facts", [])})
    if task == "policy":
        return (o.get("verdict"), sorted(o.get("cited_rules", [])), sorted(o.get("obligations", [])))
    return norm(o)

score = defaultdict(lambda: [0, 0, 0])  # n, valid JSON, match
misses = []
for row in val_rows:
    task = row.get("task", "compile")
    inputs = tok(render_prompt(row["messages"]), return_tensors="pt").to("cuda:0")  # embeddings are on GPU 0
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = json.loads(row["messages"][2]["content"])
    score[task][0] += 1
    try:
        pred = json.loads(text)
        score[task][1] += 1
        hit = key(task, pred) == key(task, gold)
        score[task][2] += hit
        if not hit:
            misses.append((task, row["id"], text, gold))
    except json.JSONDecodeError:
        misses.append((task, row["id"], text, gold))

for task, (n, valid, match) in sorted(score.items()):
    print(f"{task:8s} n={n:4d}  valid JSON {valid / n:.0%}  match {match / n:.0%}")
for task, rid, text, gold in misses[:8]:
    print("---", task, rid)
    print("pred:", text[:300])
    print("gold:", json.dumps(gold)[:300])

## Export

1. the LoRA adapter (small; keep it to re-merge later);
2. the adapter merged into the original bf16 weights, on the CPU (in `/tmp`, outside
   the 20 GB output limit);
3. GGUF with llama.cpp's own converter, then **Q5_K_M** (about 3 GB for 4B) and
   **Q4_K_M** (about 2.6 GB; the safer choice on a 6 GB laptop GPU with a long context).

Building `llama-quantize` takes a few minutes.

In [ ]:
import gc

if TRIAL:
    print("TRIAL run: set TRIAL = False and commit to train fully and export.")
else:
    model.save_pretrained(str(OUT / "lora"))
    tok.save_pretrained(str(OUT / "lora"))
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    from peft import PeftModel
    from transformers import AutoProcessor

    MERGED = Path("/tmp/merged")
    base = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, str(OUT / "lora")).merge_and_unload()
    merged.save_pretrained(MERGED)
    AutoProcessor.from_pretrained(MODEL).save_pretrained(MERGED)
    tok.save_pretrained(MERGED)
    del base, merged
    gc.collect()
    print(sorted(p.name for p in MERGED.iterdir()))

In [ ]:
if not TRIAL:
    (OUT / "gguf").mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
    !pip install -q /tmp/llama.cpp/gguf-py sentencepiece
    !python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --outtype bf16 --outfile /tmp/model-bf16.gguf
    !cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
    !cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 > /dev/null
    for q in ["Q5_K_M", "Q4_K_M"]:
        !/tmp/llama.cpp/build/bin/llama-quantize /tmp/model-bf16.gguf {OUT}/gguf/{NAME}.{q}.gguf {q}
    for f in sorted(Path("/kaggle/working").rglob("*.gguf")):
        print(f, f"{f.stat().st_size / 2**30:.2f} GB")

## Next: run it locally

Download the `.gguf` file from the version's *Output* tab, keep the 2B file beside it
to compare, then on the laptop:

```sh
llama-server -m qwen3.5-4b-nts.Q4_K_M.gguf --jinja -ngl 99 -c 8192 --port 8080
```

Thinking must stay off, as in training (the project's local client sends
`"chat_template_kwargs": {"enable_thinking": false}`). Then score it with the
`--local` evaluations, e.g. `uv run python -m evaluation.replies --parser llm --local`.